In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
timeflow_full_pipeline_with_pseudotime.py
Single-file pipeline that:
  1) loads expression, delta and metadata
  2) computes per-cell metrics: Hotspot, Specificity, Coherence
  3) normalizes metrics, saves CSV
  4) draws metric plots (3D KDE, 3D surface, spatial maps)
  5) **computes pseudotime** from the three metrics (supervised graph diffusion)
  6) runs timeflow/OT pipeline (embedding, per-bin microclusters, OT, fuse paths)
  7) produces visualizations and marker gene outputs

Configure paths and parameters in USER CONFIG block below.
"""
from __future__ import annotations
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, mannwhitneyu
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from matplotlib import cm, colors
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import Normalize
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from sklearn.exceptions import ConvergenceWarning
import networkx as nx
from scipy import sparse
from scipy.sparse.linalg import spsolve

# optional libs
try:
    import umap as _umap
    USE_UMAP = True
except Exception:
    USE_UMAP = False
try:
    import igraph as ig
    import leidenalg
    HAVE_LEIDEN = True
except Exception:
    HAVE_LEIDEN = False
try:
    import hdbscan
    HAVE_HDBSCAN = True
except Exception:
    HAVE_HDBSCAN = False
try:
    import ot
    HAVE_OT = True
except Exception:
    HAVE_OT = False
try:
    from statsmodels.stats.multitest import multipletests
    HAVE_MULTITEST = True
except Exception:
    HAVE_MULTITEST = False

warnings.filterwarnings('ignore', category=ConvergenceWarning)

# ---------------- USER CONFIG (edit as needed) ----------------
NORM_MODE = os.environ.get("SPATIAL_VELO_NORM_MODE", "2D").upper()
_CWD_DATA_DIR = os.path.join(os.getcwd(), "data", "test_data")
_DEFAULT_WORKDIR = (
    _CWD_DATA_DIR
    if os.path.isdir(_CWD_DATA_DIR)
    else os.path.join("..", "data", "test_data")
)
BASE_DIR = os.path.abspath(
    os.environ.get("SPATIAL_VELO_WORKDIR", _DEFAULT_WORKDIR)
)
BASE = BASE_DIR
EXPR_PATH = os.path.join(BASE, "transcripts_cytoplasm_matrix_common.csv")
DELTA_PATH = os.path.join(BASE, f"pipeline_out/mode_{NORM_MODE}/out_part3", "delta_export_prob_median.csv")  # use forward slash
META_PATH = os.path.join(BASE, "xenium_test_metadata.csv")

OUT_DIR = os.path.join(BASE, f"pipeline_out/mode_{NORM_MODE}/out_part3", "timeflow_full_pipeline")
OUT_METRICS = os.path.join(OUT_DIR, "metrics")
OUT_PLOTS_3D = os.path.join(OUT_METRICS, "3d_plots")
OUT_PLOTS_SPATIAL = os.path.join(OUT_METRICS, "spatial_plots")
OUT_OT = os.path.join(OUT_DIR, "ot_results")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(OUT_METRICS, exist_ok=True)
os.makedirs(OUT_PLOTS_3D, exist_ok=True)
os.makedirs(OUT_PLOTS_SPATIAL, exist_ok=True)
os.makedirs(OUT_OT, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Coordinate column names in metadata (user adjustable)
X_COORD_COL = "x"   # column name for x coordinate
Y_COORD_COL = "y"   # column name for y coordinate

# metrics / visualization params
METRIC_COLS = ["Hotspot", "Specificity", "Coherence"]
N_TIME_BINS = 5                # Number of pseudotime bins

# embedding & clustering params
EMBED_METHOD = "UMAP"   # Dimensionality reduction method: "UMAP" or "PCA"
CLUSTER_METHOD = "leiden"  # Clustering method within each time bin, choose 'leiden' | 'hdbscan' | 'kmeans'
KNN_FOR_GRAPH = 15      # Number of neighbors used for graph construction.
MIN_CLUSTERS = 5        # Minimum number of final clusters after fusion.
MAX_CLUSTERS = 15       # Maximum number of final clusters after fusion.

# OT params 
SINKHORN_REG = 0.05        #Sinkhorn regularization strength, from 0.01 (very sharp/deterministic) to 0.1-1 (very smooth/diffuse)
SINKHORN_MAXITER = 2000    #Maximum Sinkhorn iteration steps
TOP_K_EDGES = 5            #Keep the top OT edges (Controls sparsity of the OT graph)

FIG_DPI = 300
FIGSIZE = (8,8)
CMAP_METRICS = "coolwarm"      
CMAP_SPATIAL = "viridis"       

# ---------- pseudotime figure config ----------
PSEUDOTIME_POINT_SIZE = 2
PSEUDOTIME_CMAP = 'viridis'
PSEUDOTIME_MIN_COUNT = 5                     # minimal number of cells per cluster

# pseudotime early vs. late percentage (default early <0.3, late >0.7)
PSEUDOTIME_LOW_Q = 0.3
PSEUDOTIME_HIGH_Q = 0.7

# define the tendency of hotspot/specificity/coherence with time
EARLY_IDEAL = [0.0, 0.0, 0.0]     # hotspot/specificity/coherence in early stage
MIDDLE_IDEAL = [0.5, 0.5, 1.0]    # hotspot/specificity/coherence in middle stage
LATE_IDEAL = [1.0, 1.0, 0.5]      # hotspot/specificity/coherence in late stage

# pseudotime-diffusion plot parameters
PSEUDOTIME_K_NEIGHBORS = 20       #number of neighbors
PSEUDOTIME_MU = 1.0            # Smoothing weight (0-10, higher-smoother pseudotime, default 1)
PSEUDOTIME_LAB_WEIGHT = 100.0  # Confidence weight for seed cell labels 10-1000, default 100, higher = stricter fit
PSEUDOTIME_EPS = 1e-8          # Small diagonal shift to keep Laplacian invertible

# misc
MICRO_NET_TOP_EDGES = 30    #For visulization, show top-K strongest edges in the visualization graph
MICRO_NET_LINE_ALPHA = 0.6 # transparency of the line
MIN_NODE_SIZE = 80         #min size of nodes in the plot
MAX_NODE_SIZE = 800       #max size of nodes in the plot
EDGE_WIDTH_SCALE = 6       #width of the edges in the plot
CURVE_MAX_WIDTH = 6.0       #max width of the curve
CURVE_ALPHA = 0.75       #transparency of the curve
CURVE_RAD_FACTOR = 0.25  # Arc curvature for cross‑bin flow plots (0 = straight)

# ==================== LOAD AND PREPROCESS DATA ====================

def ensure_files_exist(paths):
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Required file missing: {p}")

def load_and_align(expr_path: str, delta_path: str, meta_path: str):
    expr_df = pd.read_csv(expr_path, index_col=0)
    delta_df = pd.read_csv(delta_path, index_col=0)
    meta_df = pd.read_csv(meta_path, dtype=str)
    expr_df.index = expr_df.index.astype(str)
    delta_df.index = delta_df.index.astype(str)
    expr_df.columns = expr_df.columns.astype(str)
    delta_df.columns = delta_df.columns.astype(str)

    cells = expr_df.index.intersection(delta_df.index)
    genes = expr_df.columns.intersection(delta_df.columns)
    expr_df = expr_df.loc[cells, genes].astype(float)
    delta_df = delta_df.loc[cells, genes].astype(float)

    cell_col_candidates = [c for c in meta_df.columns if 'cell' in c.lower()]
    if not cell_col_candidates:
        raise RuntimeError("Cannot detect cell ID column in metadata.")
    cell_col = cell_col_candidates[0]
    meta_df[cell_col] = meta_df[cell_col].astype(str)
    meta_df = meta_df.set_index(cell_col).loc[cells].copy()

    # Check coordinate columns
    if X_COORD_COL not in meta_df.columns:
        raise KeyError(f"X coordinate column '{X_COORD_COL}' not found in metadata. Available: {meta_df.columns.tolist()}")
    if Y_COORD_COL not in meta_df.columns:
        raise KeyError(f"Y coordinate column '{Y_COORD_COL}' not found in metadata. Available: {meta_df.columns.tolist()}")
    
    # Rename to internal names for convenience (optional, but keeps rest of code unchanged)
    meta_df.rename(columns={X_COORD_COL: 'CenterX_global_px', Y_COORD_COL: 'CenterY_global_px'}, inplace=True)
    
    return expr_df, delta_df, meta_df

def compute_cell_metrics(expr_df, delta_df, meta_df, n_neighbors=20):
    """calculate Hotspot, Specificity, Coherence"""
    cells = expr_df.index.to_numpy()
    xs = pd.to_numeric(meta_df['CenterX_global_px'], errors='coerce').to_numpy()
    ys = pd.to_numeric(meta_df['CenterY_global_px'], errors='coerce').to_numpy()
    valid_mask = (~np.isnan(xs)) & (~np.isnan(ys))
    if valid_mask.sum() == 0:
        raise RuntimeError("No valid coordinates found.")
    cells_valid = cells[valid_mask]
    xs = xs[valid_mask]; ys = ys[valid_mask]
    V = delta_df.loc[cells_valid].values.astype(float)
    mags = np.linalg.norm(V, axis=1)
    absV = np.abs(V)
    sum_abs = np.sum(absV, axis=1) + 1e-12
    specificity = np.max(absV, axis=1) / sum_abs
    coords = np.column_stack([xs, ys])
    n_samples = coords.shape[0]
    k = min(n_neighbors, max(1, n_samples-1))
    nbrs = NearestNeighbors(n_neighbors=k+1, n_jobs=-1).fit(coords)
    _, neighbors = nbrs.kneighbors(coords)
    nb_idx = neighbors[:, 1:]
    V_norm = np.linalg.norm(V, axis=1)
    V_unit = np.divide(V, V_norm[:, None], where=(V_norm[:, None] > 0))
    dots = (V_unit[nb_idx] * V_unit[:, None, :]).sum(axis=2)
    coherence = dots.mean(axis=1)
    meta_out = meta_df.loc[cells_valid].copy().reset_index()
    meta_out['Hotspot'] = mags
    meta_out['Specificity'] = specificity
    meta_out['Coherence'] = coherence
    meta_out['CenterX_global_px'] = xs
    meta_out['CenterY_global_px'] = ys
    return meta_out, cells_valid

def normalize_metrics_and_save(meta_df, out_csv_base):
    df = meta_df.copy()
    for col in METRIC_COLS:
        s = pd.to_numeric(df[col], errors='coerce')
        norm_col = col.lower() + "_n"
        if s.max() - s.min() <= 0:
            df[norm_col] = 0.0
        else:
            df[norm_col] = (s - s.min()) / (s.max() - s.min())
    out_csv = out_csv_base
    df.to_csv(out_csv, index=False)
    return out_csv, df

def plot_3d_kde(df_norm, out_png, cmap=CMAP_METRICS, dpi=FIG_DPI):
    x = df_norm["hotspot_n"].to_numpy()
    y = df_norm["specificity_n"].to_numpy()
    z = df_norm["coherence_n"].to_numpy()
    mask = (~np.isnan(x)) & (~np.isnan(y)) & (~np.isnan(z))
    x, y, z = x[mask], y[mask], z[mask]
    if x.size == 0:
        return
    positions = np.vstack([x, y, z])
    kde = gaussian_kde(positions)
    density = kde(positions)
    order = np.argsort(density)
    x_ord, y_ord, z_ord, d_ord = x[order], y[order], z[order], density[order]
    c_vmin, c_vmax = np.percentile(d_ord, [1, 99])
    norm = Normalize(vmin=c_vmin, vmax=c_vmax)
    fig = plt.figure(figsize=FIGSIZE, facecolor='none')
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(x_ord, y_ord, z_ord, c=d_ord, cmap=cmap, norm=norm, s=2, alpha=0.6, depthshade=False, edgecolors='none')
    ax.set_xlabel("Hotspot (norm, reversed)")
    ax.set_ylabel("Specificity (norm)")
    ax.set_zlabel("Coherence (norm)")
    ax.invert_xaxis()
    fig.patch.set_alpha(0); ax.set_facecolor((0,0,0,0)); ax.grid(False)
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.6)
    cbar.set_label("KDE density", rotation=270, labelpad=12)
    plt.tight_layout()
    plt.savefig(out_png, dpi=dpi, transparent=True, bbox_inches='tight', pad_inches=0.02)
    plt.close()

def plot_3d_surface(df_norm, out_png, grid_res=300, sigma=2, height_scale=0.5, cmap_name='coolwarm', dpi=FIG_DPI):
    xs = pd.to_numeric(df_norm['CenterX_global_px'], errors='coerce').values
    ys = pd.to_numeric(df_norm['CenterY_global_px'], errors='coerce').values
    coh = pd.to_numeric(df_norm['coherence_n'], errors='coerce').values
    spec = pd.to_numeric(df_norm['specificity_n'], errors='coerce').values
    mask = (~np.isnan(xs)) & (~np.isnan(ys)) & (~np.isnan(coh)) & (~np.isnan(spec))
    xs, ys, coh, spec = xs[mask], ys[mask], coh[mask], spec[mask]
    if xs.size == 0:
        return
    grid_x, grid_y = np.mgrid[xs.min():xs.max():grid_res*1j, ys.min():ys.max():grid_res*1j]
    grid_h = griddata((xs, ys), coh, (grid_x, grid_y), method='linear')
    grid_c = griddata((xs, ys), spec, (grid_x, grid_y), method='linear')
    grid_h = gaussian_filter(grid_h, sigma=sigma)
    grid_c = gaussian_filter(grid_c, sigma=sigma)
    gmin, gmax = np.nanmin(grid_h), np.nanmax(grid_h)
    if gmax - gmin <= 0:
        grid_h = np.zeros_like(grid_h)
    else:
        grid_h = (grid_h - gmin) / (gmax - gmin)
    grid_h = height_scale * grid_h
    cmin, cmax = np.nanquantile(grid_c, 0.05), np.nanquantile(grid_c, 0.95)
    norm = colors.Normalize(vmin=cmin, vmax=cmax)
    fig = plt.figure(figsize=(10,8), facecolor='none')
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(grid_x, grid_y, grid_h, facecolors=cm.get_cmap(cmap_name)(norm(grid_c)),
                    rstride=1, cstride=1, linewidth=0, antialiased=True, shade=False)
    ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)'); ax.set_zlabel('Coherence (height)')
    mappable = cm.ScalarMappable(norm=norm, cmap=cm.get_cmap(cmap_name))
    cbar = fig.colorbar(mappable, ax=ax, shrink=0.7, pad=0.1)
    cbar.set_label('Specificity (5–95%)')
    ax.set_zlim(0, 1); ax.invert_yaxis(); ax.view_init(elev=30, azim=320)
    plt.tight_layout()
    plt.savefig(out_png, dpi=dpi, transparent=True, bbox_inches='tight', pad_inches=0)
    plt.close()

def plot_spatial_metrics(df_norm, out_dir, metric_cols=('hotspot_n','specificity_n','coherence_n'),
                         cmap=CMAP_SPATIAL, lowp=5, highp=95, pt_size=2, dpi=FIG_DPI, figsize=FIGSIZE):
    xs = pd.to_numeric(df_norm['CenterX_global_px'], errors='coerce').to_numpy()
    ys = pd.to_numeric(df_norm['CenterY_global_px'], errors='coerce').to_numpy()
    valid_mask = (~np.isnan(xs)) & (~np.isnan(ys))
    xs, ys = xs[valid_mask], ys[valid_mask]
    df_valid = df_norm.loc[valid_mask].reset_index(drop=True)
    for metric in metric_cols:
        vals = pd.to_numeric(df_valid[metric], errors='coerce').to_numpy()
        if vals.size == 0:
            continue
        vmin, vmax = np.percentile(vals[~np.isnan(vals)], [lowp, highp])
        vals_clipped = np.clip(vals, vmin, vmax)
        plt.figure(figsize=figsize)
        plt.scatter(xs, ys, c=vals_clipped, s=pt_size, alpha=0.7, cmap=cmap, edgecolors='none')
        plt.axis('equal'); plt.axis('off'); plt.title(metric)
        cbar = plt.colorbar(shrink=0.7); cbar.set_label(metric)
        out_png = os.path.join(out_dir, f"{metric}_spatial_{cmap}_{lowp}-{highp}.png")
        plt.savefig(out_png, dpi=dpi, transparent=True, bbox_inches='tight', pad_inches=0.02)
        plt.close()

# ---------- compute pseudotime ----------
def compute_pseudotime_from_metrics(df_norm, out_dir,
                                    point_size=PSEUDOTIME_POINT_SIZE,
                                    cmap=PSEUDOTIME_CMAP,
                                    min_count=PSEUDOTIME_MIN_COUNT,
                                    low_q=PSEUDOTIME_LOW_Q,
                                    high_q=PSEUDOTIME_HIGH_Q,
                                    early_ideal=EARLY_IDEAL,
                                    middle_ideal=MIDDLE_IDEAL,
                                    late_ideal=LATE_IDEAL,
                                    k_neighbors=PSEUDOTIME_K_NEIGHBORS,
                                    mu=PSEUDOTIME_MU,
                                    lab_weight=PSEUDOTIME_LAB_WEIGHT,
                                    eps=PSEUDOTIME_EPS,
                                    dpi=FIG_DPI):
    df = df_norm.copy()
    for col in ["coherence_n","specificity_n","hotspot_n","CenterX_global_px","CenterY_global_px"]:
        if col not in df.columns:
            raise ValueError(f"Missing column for pseudotime: {col}")
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=["CenterX_global_px","CenterY_global_px"]).reset_index(drop=True)
    coords = df[["CenterX_global_px","CenterY_global_px"]].values
    n_cells = coords.shape[0]
    h = df["hotspot_n"].values
    s = df["specificity_n"].values
    c = df["coherence_n"].values

    def quantile_threshold(arr):
        low = np.percentile(arr, low_q*100)
        high = np.percentile(arr, high_q*100)
        low_mask = arr < low
        mid_mask = (arr >= low) & (arr <= high)
        high_mask = arr > high
        return low_mask, mid_mask, high_mask

    h_low, h_mid, h_high = quantile_threshold(h)
    s_low, s_mid, s_high = quantile_threshold(s)
    c_low, c_mid, c_high = quantile_threshold(c)

    early_mask = h_low & s_low & c_low
    middle_mask = h_mid & s_mid & c_high
    late_mask = h_high & s_high & c_mid

    def ensure_enough(mask, ideal_vec, name):
        n = int(mask.sum())
        if n >= min_count:
            return mask
        ideal = np.array(ideal_vec).reshape(1,3)
        feats = np.stack([h, s, c], axis=1)
        d = np.linalg.norm(feats - ideal, axis=1)
        ids_existing = np.where(mask)[0].tolist()
        need = max(0, min_count - len(ids_existing))
        if need == 0:
            return mask
        ids_sorted = np.argsort(d)
        pick = []
        for idx in ids_sorted:
            if idx in ids_existing:
                continue
            pick.append(idx)
            if len(pick) >= need:
                break
        new_mask = mask.copy().astype(bool)
        new_mask[pick] = True
        return new_mask

    early_mask = ensure_enough(early_mask, early_ideal, 'early')
    middle_mask = ensure_enough(middle_mask, middle_ideal, 'middle')
    late_mask = ensure_enough(late_mask, late_ideal, 'late')

    labeled_idx = np.where(early_mask | middle_mask | late_mask)[0]
    y = np.full(n_cells, np.nan)
    y[early_mask] = 0.0
    y[middle_mask] = 0.5
    y[late_mask] = 1.0

    A = kneighbors_graph(coords, n_neighbors=min(k_neighbors, n_cells-1),
                         mode='distance', include_self=False, n_jobs=-1).tocsr()
    A = 0.5 * (A + A.T)
    sigma = np.median(A.data) if A.data.size > 0 else 1.0
    if sigma <= 0: sigma = 1.0
    A.data = np.exp(- (A.data**2) / (2*sigma**2))
    W = A
    deg = np.array(W.sum(axis=1)).flatten()
    D = sparse.diags(deg)
    L = D - W + eps * sparse.eye(n_cells)
    S_diag = np.zeros(n_cells)
    S_diag[labeled_idx] = lab_weight
    S = sparse.diags(S_diag)
    rhs = S.dot(np.nan_to_num(y, nan=0.0))
    A_mat = S + mu * L
    f = spsolve(A_mat.tocsr(), rhs)
    f = np.clip(f, 0, 1)
    pseudotime = (f - f.min()) / (f.max() - f.min() + 1e-12)
    df["pseudotime_supervised_n"] = pseudotime
    pseudotime_csv = os.path.join(out_dir, "pseudotime_results.csv")
    df.to_csv(pseudotime_csv, index=False)

    fig = plt.figure(figsize=(8,8), facecolor='none')
    ax = fig.add_subplot(111, projection='3d')
    fig.patch.set_alpha(0); ax.set_facecolor((0,0,0,0)); ax.grid(False)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False
    pt_vals = pseudotime
    vmin, vmax = np.percentile(pt_vals, 5), np.percentile(pt_vals, 95)
    pt_clipped = np.clip(pt_vals, vmin, vmax)
    norm = Normalize(vmin=vmin, vmax=vmax)
    positions = np.vstack([h, s, c])
    density = gaussian_kde(positions)(positions)
    order = np.argsort(density)
    sc = ax.scatter(h[order], s[order], c[order], c=pt_clipped[order], cmap=cmap,
                    norm=norm, s=point_size, alpha=0.6, depthshade=False, edgecolors='none')
    ax.set_xlabel("Hotspot"); ax.set_ylabel("Specificity"); ax.set_zlabel("Coherence")
    ax.set_xlim(np.max(h), np.min(h))
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, aspect=10, pad=0.08)
    cbar.set_label("Pseudotime (5-95% clipped)")
    png_3d = os.path.join(out_dir, "3D_metrics_space_pseudotime_kde.png")
    plt.savefig(png_3d, dpi=dpi, transparent=True, bbox_inches='tight', pad_inches=0.02)
    plt.close()

    plt.figure(figsize=(8,8))
    sc2 = plt.scatter(coords[:,0], coords[:,1], c=pt_clipped, s=point_size+1, cmap=cmap, edgecolor='none')
    plt.gca().invert_yaxis(); plt.axis('off')
    plt.colorbar(sc2, label="Pseudotime (5-95% clipped)")
    plt.title("Spatial pseudotime (Heatmap)")
    png_space = os.path.join(out_dir, "spatial_pseudotime_heatmap.png")
    plt.savefig(png_space, dpi=dpi, transparent=True, bbox_inches='tight', pad_inches=0.02)
    plt.close()

    return df

# ---------- OT  ----------
def normalize_pseudotime(pt_series):
    pt_vals = pd.to_numeric(pt_series, errors='coerce')
    pt_norm = (pt_vals - np.nanmin(pt_vals)) / (np.nanmax(pt_vals) - np.nanmin(pt_vals) + 1e-12)
    time_bin = pd.qcut(pt_norm, q=N_TIME_BINS, labels=False, duplicates='drop')
    return pt_norm, time_bin

def scale_expression(expr_df):
    X = expr_df.values.astype(float)
    if np.median(X) > 1:
        X = np.log1p(X)
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    return Xs, scaler

def compute_embedding(Xs, method='UMAP', random_state=RANDOM_SEED):
    if method.upper() == 'UMAP' and USE_UMAP:
        umap_model = _umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=2, random_state=random_state)
        Xlow = umap_model.fit_transform(Xs)
        pca_for_velocity = PCA(n_components=10, random_state=random_state).fit(Xs)
        pca_scores = pca_for_velocity.transform(Xs)
        from sklearn.linear_model import LinearRegression
        lr = LinearRegression().fit(pca_scores, Xlow)
        A = lr.coef_
    else:
        pca = PCA(n_components=10, random_state=random_state).fit(Xs)
        pca_scores = pca.transform(Xs)
        Xlow = pca_scores[:, :2]
        pca_for_velocity = pca
        A = np.zeros((2, pca.n_components_)); A[0,0]=1.0; A[1,1]=1.0
    return Xlow, pca_for_velocity, A

def per_bin_clustering(pt_df, Xlow, method=CLUSTER_METHOD, knn_for_graph=KNN_FOR_GRAPH):
    micro_nodes = []
    cell_to_micro = np.full(len(pt_df), -1, dtype=int)
    micro_id = 0
    for b in sorted(pt_df['time_bin'].unique()):
        idx = np.where(pt_df['time_bin'].values == b)[0]
        if len(idx) == 0:
            continue
        Xb = Xlow[idx]
        if len(idx) < max(10, knn_for_graph):
            labels_b = np.zeros(len(idx), dtype=int)
        else:
            labels_b = None
            if method.lower() in ('leiden','louvain') and HAVE_LEIDEN:
                try:
                    knn = kneighbors_graph(Xb, n_neighbors=min(knn_for_graph, len(Xb)-1), include_self=False)
                    coo = knn.tocoo()
                    edges = list(zip(coo.row.tolist(), coo.col.tolist()))
                    g = ig.Graph(n=len(Xb), edges=edges, directed=False)
                    g.simplify(multiple=True, loops=True)
                    partition = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition, resolution_parameter=1.0)
                    labels_b = np.array(partition.membership, dtype=int)
                except Exception:
                    labels_b = None
            if labels_b is None and method.lower() in ('hdbscan','leiden') and HAVE_HDBSCAN:
                try:
                    clusterer = hdbscan.HDBSCAN(min_cluster_size=max(5, len(Xb)//20))
                    labels_b = clusterer.fit_predict(Xb)
                    if np.any(labels_b == -1):
                        noise = np.where(labels_b == -1)[0]
                        labeled = np.where(labels_b != -1)[0]
                        if labeled.size > 0:
                            nn = NearestNeighbors(n_neighbors=1).fit(Xb[labeled])
                            _, nns = nn.kneighbors(Xb[noise])
                            labels_b[noise] = labels_b[labeled[nns[:,0]]]
                        else:
                            labels_b = np.zeros(len(Xb), dtype=int)
                except Exception:
                    labels_b = None
            if labels_b is None:
                k_try = max(1, min(8, len(Xb)//10))
                labels_b = KMeans(n_clusters=k_try, random_state=RANDOM_SEED).fit_predict(Xb)
        for lab in np.unique(labels_b):
            members_local = np.where(labels_b == lab)[0]
            members = idx[members_local].tolist()
            centroid = Xlow[members].mean(axis=0)
            micro_nodes.append({'micro_id': micro_id, 'time_bin': int(b), 'n_cells': len(members),
                                'centroid': centroid, 'members': members, 'label_in_bin': int(lab)})
            for m in members:
                cell_to_micro[m] = micro_id
            micro_id += 1
    micro_df = pd.DataFrame([{'micro_id': m['micro_id'], 'time_bin': m['time_bin'], 'n_cells': m['n_cells'],
                              'centroid_0': float(m['centroid'][0]), 'centroid_1': float(m['centroid'][1]),
                              'label_in_bin': m['label_in_bin']} for m in micro_nodes])
    return micro_nodes, micro_df, cell_to_micro

def compute_delta_embedding(delta_df, scaler_X, pca_for_velocity, A):
    delta_arr = delta_df.values.astype(float)
    delta_scaled = scaler_X.transform(delta_arr)
    delta_pca = pca_for_velocity.transform(delta_scaled)
    try:
        delta_emb = np.dot(delta_pca, A.T)
    except Exception:
        delta_emb = delta_pca[:, :2]
    return delta_emb

def compute_ot_pairs(micro_df, top_k_edges=TOP_K_EDGES, sinkhorn_reg=SINKHORN_REG, sinkhorn_maxiter=SINKHORN_MAXITER):
    if micro_df.empty:
        return pd.DataFrame([])
    all_bins = sorted(micro_df['time_bin'].unique())
    edges = []
    for i in range(len(all_bins)-1):
        b0, b1 = all_bins[i], all_bins[i+1]
        nodes0 = micro_df[micro_df['time_bin'] == b0].reset_index(drop=True)
        nodes1 = micro_df[micro_df['time_bin'] == b1].reset_index(drop=True)
        if nodes0.empty or nodes1.empty:
            continue
        cent0 = nodes0[['centroid_0','centroid_1']].values
        cent1 = nodes1[['centroid_0','centroid_1']].values
        C = np.linalg.norm(cent0[:,None,:] - cent1[None,:,:], axis=2)
        a_raw = nodes0['n_cells'].values.astype(float)
        b_raw = nodes1['n_cells'].values.astype(float)
        a = a_raw / (a_raw.sum() + 1e-12)
        b = b_raw / (b_raw.sum() + 1e-12)
        if not HAVE_OT:
            raise RuntimeError('POT (ot) not installed')
        try:
            Gmat = ot.sinkhorn(a, b, C, reg=sinkhorn_reg, numItermax=sinkhorn_maxiter)
        except Exception:
            Gmat = ot.emd(a, b, C)
        Gcells = Gmat * a_raw.sum()
        for si in range(Gmat.shape[0]):
            row = Gmat[si,:]
            top_idx = np.argsort(row)[::-1][:top_k_edges]
            for tj in top_idx:
                flow_cells = float(Gcells[si, tj])
                if flow_cells <= 0:
                    continue
                edges.append({'src_micro': int(nodes0.loc[si,'micro_id']), 'tgt_micro': int(nodes1.loc[tj,'micro_id']),
                              'src_bin': int(b0), 'tgt_bin': int(b1), 'flow_frac': float(row[tj]),
                              'flow_cells': flow_cells, 'src_n': int(nodes0.loc[si,'n_cells']), 'tgt_n': int(nodes1.loc[tj,'n_cells'])})
    edges_df = pd.DataFrame(edges)
    if not edges_df.empty:
        bin_totals = micro_df.groupby('time_bin')['n_cells'].sum().to_dict()
        edges_df['pct_of_src_micro'] = edges_df.apply(lambda r: 0.0 if r['src_n']==0 else (r['flow_cells']/r['src_n'])*100.0, axis=1)
        edges_df['pct_of_src_bin'] = edges_df.apply(lambda r: 0.0 if bin_totals.get(r['src_bin'],0)==0 else (r['flow_cells']/bin_totals[r['src_bin']])*100.0, axis=1)
    return edges_df

def build_temporal_graph(edges_df):
    G = nx.DiGraph()
    if edges_df.empty:
        return G
    nodes = np.unique(np.concatenate([edges_df['src_micro'].unique(), edges_df['tgt_micro'].unique()]))
    for n in nodes:
        G.add_node(int(n))
    for _, r in edges_df.iterrows():
        s, t, w = int(r['src_micro']), int(r['tgt_micro']), float(r['flow_cells'])
        if G.has_edge(s,t):
            G[s][t]['weight'] += w
        else:
            G.add_edge(s,t, weight=w)
    return G

def extract_temporal_paths(G, micro_df, min_path_len=1):
    if G.number_of_nodes() == 0:
        return []
    time_map = {int(r['micro_id']): int(r['time_bin']) for _, r in micro_df.iterrows()}
    bins_present = sorted(set(time_map[n] for n in G.nodes() if n in time_map))
    if not bins_present:
        return []
    seed_bin = bins_present[0]
    seeds = [n for n in G.nodes() if time_map.get(n, 999) == seed_bin]
    used = set()
    paths = []
    for s in seeds:
        if s in used: continue
        path = [s]; used.add(s); cur = s
        while True:
            outs = [(v, G[cur][v]['weight']) for v in G.successors(cur)]
            outs = [(v,w) for (v,w) in outs if time_map.get(v, 999) > time_map.get(cur, -1)]
            if not outs: break
            v, w = max(outs, key=lambda x: x[1])
            if v in used: break
            path.append(v); used.add(v); cur = v
        if len(path) >= min_path_len:
            paths.append(path)
    remaining = [n for n in G.nodes() if n not in used]
    for n in remaining:
        preds = [(p, G[p][n]['weight']) for p in G.predecessors(n) if p in used]
        if preds:
            best_p, _ = max(preds, key=lambda x: x[1])
            for path in paths:
                if best_p in path:
                    last_time = micro_df.set_index('micro_id').loc[path[-1],'time_bin']
                    if micro_df.set_index('micro_id').loc[n,'time_bin'] > last_time:
                        path.append(n); used.add(n); break
    remaining2 = [n for n in G.nodes() if n not in used]
    for n in remaining2:
        paths.append([n])
    return paths

def fuse_paths_to_clusters(paths, micro_nodes, Xlow, min_clusters=MIN_CLUSTERS, max_clusters=MAX_CLUSTERS):
    micro_to_final = {}
    for i, p in enumerate(paths):
        for mid in p:
            micro_to_final[mid] = i
    micro_map = {m['micro_id']: m for m in micro_nodes}
    def cluster_size(cid):
        mids = [mid for mid,c in micro_to_final.items() if c==cid]
        return sum(micro_map[mid]['n_cells'] for mid in mids)
    def cluster_centroid(cid):
        mids = [mid for mid,c in micro_to_final.items() if c==cid]
        pts=[]; ws=[]
        for mid in mids:
            rec = micro_map[mid]
            pts.append(rec['centroid']); ws.append(rec['n_cells'])
        if not pts: return np.array([0.0,0.0])
        pts = np.array(pts); ws = np.array(ws, dtype=float)
        return np.average(pts, axis=0, weights=ws)
    # Merge until <= max_clusters
    while len(set(micro_to_final.values())) > max_clusters:
        cids = sorted(list(set(micro_to_final.values())))
        sizes = {c: cluster_size(c) for c in cids}
        small = min(sizes.items(), key=lambda x: x[1])[0]
        cent_small = cluster_centroid(small)
        others = [c for c in cids if c!=small]
        dmin=None; nearest=None
        for o in others:
            d = np.linalg.norm(cent_small - cluster_centroid(o))
            if dmin is None or d < dmin:
                dmin = d; nearest = o
        for mid in list(micro_to_final.keys()):
            if micro_to_final[mid] == small:
                micro_to_final[mid] = nearest
    # If too few clusters, split largest by KMeans on member cells
    if len(set(micro_to_final.values())) < min_clusters:
        next_cid = max(micro_to_final.values())+1 if micro_to_final else 0
        while len(set(micro_to_final.values())) < min_clusters:
            sizes = {c: cluster_size(c) for c in set(micro_to_final.values())}
            largest = max(sizes.items(), key=lambda x: x[1])[0]
            mids = [mid for mid,c in micro_to_final.items() if c==largest]
            cell_idxs = []
            for mid in mids:
                cell_idxs.extend(micro_map[mid]['members'])
            cell_idxs = np.array(cell_idxs, dtype=int)
            if len(cell_idxs) < 10: break
            feat = Xlow[cell_idxs]
            try:
                km = KMeans(n_clusters=2, random_state=RANDOM_SEED).fit(feat)
            except Exception:
                break
            labs = km.labels_
            new_cid = next_cid; next_cid += 1
            for mid in mids:
                members = np.array(micro_map[mid]['members'], dtype=int)
                if members.size == 0: continue
                mask = np.isin(cell_idxs, members)
                if not mask.any(): continue
                member_labels = labs[mask]
                if np.mean(member_labels) > 0.5:
                    micro_to_final[mid] = new_cid
                else:
                    micro_to_final[mid] = largest
            if len(set(micro_to_final.values())) >= min_clusters:
                break
    return micro_to_final

def assign_cell_final_labels(cell_to_micro, micro_to_final):
    if not micro_to_final:
        return np.full(cell_to_micro.shape[0], -1, dtype=int)
    max_mid = max(micro_to_final.keys())
    map_arr = np.full(max_mid+1, -1, dtype=int)
    for mid, fid in micro_to_final.items():
        map_arr[mid] = fid
    final_labels = np.full(cell_to_micro.shape[0], -1, dtype=int)
    mask = (cell_to_micro >= 0) & (cell_to_micro <= max_mid)
    final_labels[mask] = map_arr[cell_to_micro[mask]]
    return final_labels

def prepare_cluster_colors(cluster_ids):
    cmap = plt.get_cmap('tab20')
    unique = np.unique(cluster_ids)
    unique = unique[unique >= 0]
    cluster_color = {}
    for i, c in enumerate(unique):
        cluster_color[int(c)] = cmap(i % cmap.N)
    cluster_color[-1] = (0.7,0.7,0.7,0.4)
    return cluster_color

def compute_markers_and_top_genes(expr_df, pt_df, out_dir, top_n=10):
    gene_names = expr_df.columns.tolist()
    expr_array = expr_df.values
    clusters = np.unique(pt_df['global_cluster'].values)
    clusters = clusters[clusters>=0]
    marker_rows = []
    top_genes_by_cluster = {}
    for c in clusters:
        idx_in = np.where(pt_df['global_cluster'].values==c)[0]
        idx_out = np.where(pt_df['global_cluster'].values!=c)[0]
        if idx_in.size < 3 or idx_out.size < 3:
            top_genes_by_cluster[int(c)] = []
            continue
        pvals = np.ones(len(gene_names))
        mean_in = np.mean(expr_array[idx_in,:], axis=0)
        mean_out = np.mean(expr_array[idx_out,:], axis=0)
        for gi in range(len(gene_names)):
            try:
                _, p = mannwhitneyu(expr_array[idx_in,gi], expr_array[idx_out,gi], alternative='two-sided')
            except Exception:
                p = 1.0
            pvals[gi] = p
        if HAVE_MULTITEST:
            _, adj_pvals, _, _ = multipletests(pvals, method='fdr_bh')
        else:
            order = np.argsort(pvals)
            ranked = np.empty_like(order)
            ranked[order] = np.arange(1, len(pvals)+1)
            adj = pvals * len(pvals) / ranked
            adj_pvals = np.minimum.accumulate(adj[np.argsort(order)])[np.argsort(order)]
            adj_pvals = np.minimum(adj_pvals, 1.0)
        logfc = np.log2((mean_in + 1e-9) / (mean_out + 1e-9))
        df_mark = pd.DataFrame({'cluster': int(c), 'gene': gene_names, 'pval': pvals, 'adj_pval': adj_pvals,
                                'mean_in': mean_in, 'mean_out': mean_out, 'log2FC': logfc})
        df_mark = df_mark.sort_values(['adj_pval','log2FC'], ascending=[True, False]).reset_index(drop=True)
        top10 = df_mark.head(top_n)
        top_genes_by_cluster[int(c)] = top10['gene'].tolist()
        marker_rows.append(df_mark.head(100))
    markers_all = pd.concat(marker_rows, ignore_index=True) if marker_rows else pd.DataFrame()
    markers_all.to_csv(os.path.join(out_dir, 'cluster_markers_top100_aggregate.csv'), index=False)
    rows = []
    for c, genes in top_genes_by_cluster.items():
        for rank, g in enumerate(genes[:top_n], start=1):
            rows.append({'cluster': c, 'rank': rank, 'gene': g})
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, 'cluster_markers_top10.csv'), index=False)
    return top_genes_by_cluster

# ---------- Visualization ----------
def plot_bin_pair_arc_flows(edges_df, out_dir, all_bins):
    if edges_df.empty:
        return
    bin_pair_sum = edges_df.groupby(['src_bin','tgt_bin'])['flow_cells'].sum().reset_index()
    if bin_pair_sum.empty:
        return
    fig, ax = plt.subplots(figsize=(10,4))
    n_bins = len(all_bins)
    x = np.arange(n_bins)
    bin_pos = {b:i for i,b in enumerate(all_bins)}
    for xi, b in enumerate(all_bins):
        ax.scatter(xi, 0, s=200, color='white', edgecolor='k', zorder=3)
        ax.text(xi, -0.05, f'Bin {b}', ha='center', va='top')
    maxflow = bin_pair_sum['flow_cells'].max()
    for _, r in bin_pair_sum.iterrows():
        i = bin_pos[int(r['src_bin'])]; j = bin_pos[int(r['tgt_bin'])]
        flow_cells = float(r['flow_cells'])
        if i == j:
            continue
        dist = j - i
        rad = CURVE_RAD_FACTOR * np.sign(dist) * abs(dist)
        width = 0.5 + (flow_cells / maxflow) * CURVE_MAX_WIDTH
        arrow = FancyArrowPatch((i, 0), (j, 0), connectionstyle=f"arc3,rad={rad}",
                                arrowstyle='-|>', mutation_scale=12 + (flow_cells/maxflow)*8,
                                linewidth=width, color='steelblue', alpha=CURVE_ALPHA)
        ax.add_patch(arrow)
        mid_x = (i + j) / 2.0
        mid_y = 0.08 + 0.06 * abs(dist)
        ax.text(mid_x, mid_y, f"{int(round(flow_cells))} cells", ha='center', va='bottom', fontsize=9)
    ax.set_xlim(-0.5, n_bins-0.5); ax.set_ylim(-1.0, 1.0)
    ax.axis('off'); ax.set_title('Aggregate OT flows between bins (cells annotated)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'bin_pair_arc_flows_annotated.png'), dpi=200)
    plt.close()

def micro_flow_network_two_views(micro_df, edges_df, micro_nodes, pt_df, out_dir, top_edges=MICRO_NET_TOP_EDGES):
    if micro_df.empty or edges_df.empty:
        return
    edges_sorted = edges_df.sort_values('flow_cells', ascending=False).reset_index(drop=True)
    edges_top = edges_sorted.head(max(top_edges, 1))
    nodes_keep = set(edges_top['src_micro'].astype(int).tolist() + edges_top['tgt_micro'].astype(int).tolist())
    micro_sub = micro_df[micro_df['micro_id'].isin(nodes_keep)].copy()
    pos = {int(r['micro_id']): (float(r['centroid_0']), float(r['centroid_1'])) for _, r in micro_sub.iterrows()}
    node_list = sorted(pos.keys())
    ncells = np.array([int(micro_df.set_index('micro_id').loc[n,'n_cells']) if n in micro_df['micro_id'].values else 1 for n in node_list], dtype=float)
    min_nc, max_nc = ncells.min(), ncells.max()
    if max_nc == min_nc:
        node_sizes = np.full_like(ncells, (MIN_NODE_SIZE + MAX_NODE_SIZE)/2)
    else:
        node_sizes = MIN_NODE_SIZE + (ncells - min_nc) / (max_nc - min_nc) * (MAX_NODE_SIZE - MIN_NODE_SIZE)
    micro_members_map = {m['micro_id']: np.array(m['members'], dtype=int) for m in micro_nodes}
    node_cluster_major = []
    node_pt_mean = []
    for n in node_list:
        members = micro_members_map.get(int(n), np.array([], dtype=int))
        if members.size == 0:
            node_cluster_major.append(-1); node_pt_mean.append(np.nan)
        else:
            labs, counts = np.unique(pt_df['global_cluster'].values[members], return_counts=True)
            dom = int(labs[np.argmax(counts)]) if labs.size>0 else -1
            node_cluster_major.append(dom)
            node_pt_mean.append(float(np.mean(pt_df['pt_norm'].values[members])))
    node_pt_mean = np.array(node_pt_mean)
    cluster_color_map = prepare_cluster_colors(np.array(node_cluster_major))

    import matplotlib as mpl
    cmap_pt = plt.get_cmap('viridis')
    if np.any(~np.isnan(node_pt_mean)):
        vmin = np.nanpercentile(node_pt_mean, 5)
        vmax = np.nanpercentile(node_pt_mean, 95)
        if vmin == vmax:
            vmin, vmax = np.nanmin(node_pt_mean), np.nanmax(node_pt_mean) + 1e-6
    else:
        vmin, vmax = 0.0, 1.0
    norm_pt = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    for view in ('by_cluster', 'by_pseudotime'):
        fig, ax = plt.subplots(figsize=(12,9))
        maxedge = edges_top['flow_cells'].max() if not edges_top.empty else 1.0
        for _, r in edges_top.iterrows():
            s = int(r['src_micro']); t = int(r['tgt_micro']); w = float(r['flow_cells'])
            if s not in pos or t not in pos: continue
            p0 = pos[s]; p1 = pos[t]
            wnorm = w / maxedge
            lw = 1.0 + EDGE_WIDTH_SCALE * wnorm
            arrow = FancyArrowPatch(p0, p1, connectionstyle="arc3,rad=0.0", arrowstyle='-|>',
                                    mutation_scale=10, linewidth=lw, color='black', alpha=MICRO_NET_LINE_ALPHA)
            ax.add_patch(arrow)
        xs = [pos[n][0] for n in node_list]; ys = [pos[n][1] for n in node_list]
        if view == 'by_cluster':
            cols = [cluster_color_map.get(int(c), cluster_color_map.get(-1)) for c in node_cluster_major]
        else:
            cols = [cmap_pt(norm_pt(v)) if not np.isnan(v) else (0.8,0.8,0.8,0.9) for v in node_pt_mean]
        ax.scatter(xs, ys, s=node_sizes, c=cols, edgecolors='k', alpha=0.95)
        margin_x = (max(xs)-min(xs))*0.03 if len(xs)>1 else 0.1
        margin_y = (max(ys)-min(ys))*0.03 if len(ys)>1 else 0.1
        ax.set_xlim(min(xs)-margin_x, max(xs)+margin_x); ax.set_ylim(min(ys)-margin_y, max(ys)+margin_y)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'Microcluster OT Flow Network (top {len(edges_top)} edges) - {view}')
        if view == 'by_pseudotime':
            sm = mpl.cm.ScalarMappable(norm=norm_pt, cmap=cmap_pt)
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label('micro mean pt_norm (5-95% clipped)')
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'micro_flow_network_{view}.png'), dpi=300)
        plt.close()

def plot_3d_time_cluster(pt_df, out_dir, cluster_color):
    try:
        fig = plt.figure(figsize=(9,7))
        ax = fig.add_subplot(111, projection='3d')
        fig.patch.set_alpha(0.0); ax.set_facecolor((0,0,0,0)); ax.grid(False)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            try:
                axis.pane.set_facecolor((1,1,1,0)); axis.pane.set_edgecolor((1,1,1,0))
            except Exception:
                pass
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        cell_colors = [cluster_color.get(int(c), cluster_color.get(-1)) for c in pt_df['global_cluster'].values]
        ax.scatter(pt_df['pt_norm'].values, pt_df['emb1'].values, pt_df['emb2'].values,
                   c=cell_colors, s=4, alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, '3D_time_cluster.png'), dpi=300, transparent=True)
        plt.close()
    except Exception as e:
        print(f'3D plotting failed: {e}')

def plot_cluster_composition_bars(pt_df, out_dir):
    counts = pd.crosstab(pt_df['time_bin'], pt_df['global_cluster'])
    fracs = counts.div(counts.sum(axis=1), axis=0).fillna(0)
    bins = fracs.index.values
    clusters = fracs.columns.values
    cmap = plt.get_cmap('tab20')
    bottom = np.zeros(len(bins))
    plt.figure(figsize=(8,5))
    for j, c in enumerate(clusters):
        vals = fracs[c].values
        plt.bar(bins, vals, bottom=bottom, color=cmap(j % cmap.N), width=0.85, label=f'C{int(c)}')
        bottom += vals
    plt.xlabel('Time bin'); plt.ylabel('Cluster fraction'); plt.title('Cluster composition per time bin')
    plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'bin_cluster_fraction_bars.png'), dpi=300)
    plt.close()

def plot_delta_quiver(micro_nodes, delta_emb, pt_df, micro_df_final, out_dir, xlims=None, ylims=None):
    heat_X, heat_Y, heat_U, heat_V, micro_ids, micro_n = [], [], [], [], [], []
    for m in micro_nodes:
        members = np.array(m['members'], dtype=int)
        if members.size == 0: continue
        heat_X.append(float(m['centroid'][0])); heat_Y.append(float(m['centroid'][1]))
        heat_U.append(float(np.mean(delta_emb[members,0]))); heat_V.append(float(np.mean(delta_emb[members,1])))
        micro_ids.append(int(m['micro_id'])); micro_n.append(int(m['n_cells']))
    fig, ax = plt.subplots(figsize=(8,6))
    if len(heat_X) > 0:
        ax.quiver(heat_X, heat_Y, heat_U, heat_V, angles='xy', scale_units='xy', scale=1.5, width=0.003, alpha=0.85)
    micro_members_map = {m['micro_id']: np.array(m['members'], dtype=int) for m in micro_nodes}
    micro_pt_mean = []
    micro_final_cluster = []
    for mid in micro_ids:
        members = micro_members_map.get(int(mid), np.array([], dtype=int))
        if members.size == 0:
            micro_pt_mean.append(0.0); micro_final_cluster.append(-1)
        else:
            micro_pt_mean.append(float(np.mean(pt_df['pt_norm'].values[members])))
            if micro_df_final is not None and int(mid) in micro_df_final['micro_id'].values:
                micro_final_cluster.append(int(micro_df_final.set_index('micro_id').loc[mid,'final_cluster']))
            else:
                micro_final_cluster.append(-1)
    micro_pt_mean = np.array(micro_pt_mean)
    if np.any(~np.isnan(micro_pt_mean)):
        vmin = np.nanpercentile(micro_pt_mean, 5); vmax = np.nanpercentile(micro_pt_mean, 95)
        if vmin == vmax:
            vmin, vmax = micro_pt_mean.min(), micro_pt_mean.max() + 1e-6
    else:
        vmin, vmax = 0.0, 1.0
    import matplotlib as mpl
    cmap = plt.get_cmap('viridis')
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cluster_color_map = prepare_cluster_colors(np.array(micro_final_cluster))
    nc = np.array(micro_n, dtype=float)
    if nc.size == 0:
        nc_size = np.array([])
    else:
        nc_min, nc_max = nc.min(), nc.max()
        if nc_max == nc_min:
            nc_size = np.full_like(nc, 50.0)
        else:
            nc_size = 30.0 + (nc - nc_min) / (nc_max - nc_min) * 150.0
    for xi, yi, ptv, mid, sz, fc in zip(heat_X, heat_Y, micro_pt_mean, micro_ids, nc_size, micro_final_cluster):
        facec = cmap(norm(ptv))
        edgec = cluster_color_map.get(int(fc), cluster_color_map.get(-1))
        ax.scatter(xi, yi, s=sz, facecolors=[facec], edgecolors=[edgec], linewidths=0.8, alpha=0.95, zorder=4)
    ax.set_title('Delta-based flow heatmap (microcluster average velocities) with pt_norm (fill) and final-cluster (edge)')
    if xlims is not None: ax.set_xlim(xlims)
    if ylims is not None: ax.set_ylim(ylims)
    ax.set_aspect('equal', adjustable='box'); ax.set_xticks([]); ax.set_yticks([])
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('micro mean pt_norm (5-95% clipped)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'delta_flow_heatmap_quiver_with_scatter.png'), dpi=200)
    plt.close()

def plot_top3_genes_3d(expr_df, pt_df, top_genes_by_cluster, out_dir, embed_name='UMAP'):
    for c, genes in top_genes_by_cluster.items():
        top_genes = genes[:3]
        if len(top_genes) == 0:
            continue
        fig = plt.figure(figsize=(15,5))
        for i, gene in enumerate(top_genes):
            ax = fig.add_subplot(1,3,i+1, projection='3d')
            if gene not in expr_df.columns:
                vals = np.zeros(len(pt_df))
            else:
                vals = expr_df[gene].values.astype(float)
            vmin = np.nanpercentile(vals, 5) if np.any(~np.isnan(vals)) else 0.0
            vmax = np.nanpercentile(vals, 95) if np.any(~np.isnan(vals)) else 1.0
            if vmin == vmax:
                vmin = np.nanmin(vals) if np.any(~np.isnan(vals)) else vmin
                vmax = vmin + 1e-6
            sc = ax.scatter(pt_df['pt_norm'].values, pt_df['emb1'].values, pt_df['emb2'].values,
                            c=vals, s=10, alpha=0.6, cmap='viridis', vmin=vmin, vmax=vmax, edgecolors='none')
            ax.set_xlabel('pseudotime'); ax.set_ylabel(f'{embed_name}1'); ax.set_zlabel(f'{embed_name}2')
            ax.set_title(f'Cluster {c} top gene {i+1}: {gene}')
            ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
            cbar = plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.05)
            cbar.set_label('expression (5-95% clipped)')
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'cluster_{c}_top3_genes_3D.png'), dpi=200)
        plt.close()

def plot_per_bin_microclusters(pt_df, out_dir, cluster_color, xlims=None, ylims=None):
    all_x = pt_df['emb1'].values; all_y = pt_df['emb2'].values
    for b in sorted(pt_df['time_bin'].unique()):
        fig, ax = plt.subplots(figsize=(7,6))
        ax.scatter(all_x, all_y, s=6, c='lightgray', alpha=0.15, edgecolors='none')
        dfb = pt_df[pt_df['time_bin'] == b]
        if len(dfb) > 0:
            cols = [cluster_color.get(int(x), cluster_color.get(-1)) for x in dfb['global_cluster'].values]
            ax.scatter(dfb['emb1'], dfb['emb2'], s=18, c=cols, alpha=0.7, edgecolors='none')
        ax.set_title(f"Time bin {b}")
        if xlims: ax.set_xlim(xlims)
        if ylims: ax.set_ylim(ylims)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'per_bin_microclusters_bin{b}.png'), dpi=200)
        plt.close()

# ==================== MAIN ====================
def run_pipeline():
    t0 = time.time()
    ensure_files_exist([EXPR_PATH, DELTA_PATH, META_PATH])
    print("Loading and aligning data...")
    expr_df, delta_df, meta_df = load_and_align(EXPR_PATH, DELTA_PATH, META_PATH)

    # STEP1
    print("Computing cell-level metrics (Hotspot, Specificity, Coherence)...")
    meta_metrics, valid_cells = compute_cell_metrics(expr_df, delta_df, meta_df, n_neighbors=20)
    metrics_csv = os.path.join(OUT_METRICS, "cell_metrics.csv")
    meta_metrics.to_csv(metrics_csv, index=False)
    print(f"Saved raw metrics: {metrics_csv}")

    #STEP2
    out_norm_csv = os.path.join(OUT_METRICS, "cell_metrics_with_norm.csv")
    _, df_norm = normalize_metrics_and_save(meta_metrics, out_norm_csv)
    print(f"Saved normalized metrics: {out_norm_csv}")

    #STEP3
    print("Plotting metrics (3D KDE / 3D surface / spatial maps)...")
    plot_3d_kde(df_norm, os.path.join(OUT_PLOTS_3D, "3d_metrics_norm_density_hotspot_reversed.png"))
    plot_3d_surface(df_norm, os.path.join(OUT_PLOTS_3D, "3d_surface_coherence_height_specificity_color.png"))
    plot_spatial_metrics(df_norm, OUT_PLOTS_SPATIAL, metric_cols=('hotspot_n','specificity_n','coherence_n'), cmap=CMAP_SPATIAL)

    #STEP4
    print("Computing pseudotime from Hotspot/Specificity/Coherence metrics...")
    df_with_pt = compute_pseudotime_from_metrics(df_norm, out_dir=OUT_DIR)

    #STEP5
    cell_to_pt = dict(zip(df_with_pt['index'], df_with_pt['pseudotime_supervised_n']))
    common_cells = np.intersect1d(valid_cells, list(cell_to_pt.keys()))
    if len(common_cells) == 0:
        raise RuntimeError("No overlapping cells between pseudotime and metric-valid cells.")
    expr_df_sub = expr_df.loc[common_cells]
    delta_df_sub = delta_df.loc[common_cells]
    pt_df = pd.DataFrame({'cell': common_cells, 'pseudotime_raw': [cell_to_pt[c] for c in common_cells]})
    pt_df = pt_df.set_index('cell')
    pt_norm, time_bin = normalize_pseudotime(pt_df['pseudotime_raw'])
    pt_df['pt_norm'] = pt_norm
    pt_df['time_bin'] = time_bin

    print("Running timeflow/OT pipeline...")
    Xs, scaler = scale_expression(expr_df_sub)
    Xlow, pca_for_velocity, A = compute_embedding(Xs, method=EMBED_METHOD)
    pt_df['emb1'] = Xlow[:,0]; pt_df['emb2'] = Xlow[:,1]

    micro_nodes, micro_df, cell_to_micro = per_bin_clustering(pt_df, Xlow, method=CLUSTER_METHOD, knn_for_graph=KNN_FOR_GRAPH)
    micro_df.to_csv(os.path.join(OUT_OT, 'micro_nodes_pre_fusion.csv'), index=False)
    pd.DataFrame({'cell': pt_df.index.to_numpy(), 'time_bin': pt_df['time_bin'].values, 'pt_norm': pt_df['pt_norm'].values,
                  'emb1': pt_df['emb1'].values, 'emb2': pt_df['emb2'].values, 'micro_id_pre': cell_to_micro}).to_csv(os.path.join(OUT_OT, 'cell_to_micro_pre_fusion.csv'), index=False)

    delta_emb = compute_delta_embedding(delta_df_sub, scaler, pca_for_velocity, A)
    edges_df = compute_ot_pairs(micro_df)
    edges_df.to_csv(os.path.join(OUT_OT, 'micro_edges_ot.csv'), index=False)

    G = build_temporal_graph(edges_df)
    paths = extract_temporal_paths(G, micro_df)
    micro_to_final = fuse_paths_to_clusters(paths, micro_nodes, Xlow)
    final_labels = assign_cell_final_labels(cell_to_micro, micro_to_final)
    pt_df['global_cluster'] = final_labels

    micro_df_final = pd.DataFrame([{'micro_id': m['micro_id'], 'time_bin': m['time_bin'], 'n_cells': m['n_cells'],
                                    'centroid_0': float(m['centroid'][0]), 'centroid_1': float(m['centroid'][1]),
                                    'label_in_bin': m['label_in_bin'], 'final_cluster': int(micro_to_final.get(m['micro_id'], -1))}
                                   for m in micro_nodes])
    micro_df_final.to_csv(os.path.join(OUT_OT, 'micro_nodes.csv'), index=False)
    cell_map_df = pd.DataFrame({'cell': pt_df.index.to_numpy(), 'time_bin': pt_df['time_bin'].values,
                                'pt_norm': pt_df['pt_norm'].values, 'emb1': pt_df['emb1'].values,
                                'emb2': pt_df['emb2'].values, 'micro_id': cell_to_micro, 'final_cluster': pt_df['global_cluster'].values})
    cell_map_df.to_csv(os.path.join(OUT_OT, 'cell_to_micro.csv'), index=False)
    print(f'Final global clusters: {len(np.unique(final_labels[final_labels>=0]))}')

    cluster_color = prepare_cluster_colors(pt_df['global_cluster'].values)
    x_min, x_max = float(Xlow[:,0].min()), float(Xlow[:,0].max())
    y_min, y_max = float(Xlow[:,1].min()), float(Xlow[:,1].max())
    pad_x = (x_max - x_min) * 0.05 if (x_max - x_min) > 0 else 1.0
    pad_y = (y_max - y_min) * 0.05 if (y_max - y_min) > 0 else 1.0
    xlims = (x_min - pad_x, x_max + pad_x); ylims = (y_min - pad_y, y_max + pad_y)

    # PLOT
    plot_per_bin_microclusters(pt_df, OUT_OT, cluster_color, xlims=xlims, ylims=ylims)
    if not edges_df.empty:
        plot_bin_pair_arc_flows(edges_df, OUT_OT, sorted(micro_df['time_bin'].unique()))
    plot_delta_quiver(micro_nodes, delta_emb, pt_df, micro_df_final, OUT_OT, xlims=xlims, ylims=ylims)
    micro_flow_network_two_views(micro_df_final, edges_df, micro_nodes, pt_df, OUT_OT, top_edges=MICRO_NET_TOP_EDGES)
    plot_3d_time_cluster(pt_df, OUT_OT, cluster_color)
    plot_cluster_composition_bars(pt_df, OUT_OT)

    # MARKER GENE
    print("Computing cluster markers and top genes...")
    top_genes_by_cluster = compute_markers_and_top_genes(expr_df_sub, pt_df, OUT_OT, top_n=10)
    embed_name = 'UMAP' if EMBED_METHOD.upper()=='UMAP' and USE_UMAP else 'PC'
    plot_top3_genes_3d(expr_df_sub, pt_df, top_genes_by_cluster, OUT_OT, embed_name=embed_name)

    t1 = time.time()
    print(f"Pipeline done. Outputs saved to: {OUT_DIR} (elapsed {t1 - t0:.1f}s)")

if __name__ == "__main__":
    run_pipeline()